# 02 — Precipitation Interpolation

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | Open-Meteo archive grid (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · RBF · Spline |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live Open-Meteo data.  
**🛰 GEE cells** validate against CHIRPS (requires `earthengine authenticate`).

In [ ]:
%pip install -q "geointerpo[kriging,viz,data,raster]"

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY  = (-10.0, 35.0, 30.0, 55.0)  # Western Europe / Mediterranean
DATE      = '2024-01-10'
RESOLUTION = 0.5  # degrees (~50 km)

---
## Part A — Offline demo (no network needed)

In [ ]:
result = Pipeline(
    data='sample',                # Step 1 — built-in synthetic precipitation data
    variable='precipitation',
    boundary=BOUNDARY,            # Step 2 — four corners
    method=['idw', 'kriging', 'rbf', 'spline'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'gaussian'},
        'rbf':    {'kernel': 'thin_plate_spline'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
result.metrics_table()

In [ ]:
result.plot(cmap='Blues')
plt.suptitle('Precipitation — synthetic data (offline demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='Blues', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('Precipitation stations (synthetic, mm/day)')
plt.tight_layout()

### Effect of network sparsity

Re-run with progressively fewer stations to see how CV metrics degrade.

In [ ]:
from geointerpo.data.samples import load_precipitation
from geointerpo.interpolators import KrigingInterpolator

full_gdf = load_precipitation(bbox=BOUNDARY)
rows = []
for n in [50, 30, 20, 15, 10]:
    sparse = full_gdf.sample(min(n, len(full_gdf)), random_state=42).reset_index(drop=True)
    model = KrigingInterpolator(variogram_model='gaussian').fit(sparse)
    cv = model.cross_validate(sparse, k=5)
    rows.append({'n_stations': n, **{k: round(v, 3) for k, v in cv.items() if k != 'n'}})

pd.DataFrame(rows).set_index('n_stations')

---
## Part B — 🌐 Live Open-Meteo data

Open-Meteo provides a free archive grid — no API key required.

In [ ]:
result_live = Pipeline(
    data='openmeteo',              # Step 1 — live Open-Meteo archive
    variable='precipitation',
    date=DATE,
    boundary=BOUNDARY,             # Step 2 — four corners
    method=['idw', 'kriging', 'rbf'],  # Step 3
    method_params={
        'kriging': {'variogram_model': 'gaussian'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print(f'Loaded {len(result_live.stations)} grid points')

In [ ]:
# Sub-sample to simulate a sparse rain-gauge network
sparse_gdf = result_live.stations.sample(25, random_state=42).reset_index(drop=True)
print(f'Sparse network: {len(sparse_gdf)} stations')

from geointerpo.interpolators import KrigingInterpolator
from geointerpo.boundaries import boundary_bbox
from geointerpo.boundaries import load_boundary

bnd = load_boundary(BOUNDARY)
bbox = boundary_bbox(bnd)
model = KrigingInterpolator(variogram_model='gaussian').fit(sparse_gdf)
grid  = model.predict(bbox, resolution=RESOLUTION)
cv    = model.cross_validate(sparse_gdf, k=5)
print(f'Sparse Kriging CV  RMSE={cv["rmse"]:.2f}  r={cv["r"]:.3f}')

In [ ]:
result_live.plot(cmap='Blues')
plt.suptitle(f'Precipitation — {DATE}  (Open-Meteo)', y=1.02)
plt.show()

### Interactive map

In [ ]:
# Interactive map — requires: pip install 'geointerpo[notebooks]'
# (leafmap + localtileserver are notebook-only tools, not part of the core library)
try:
    import leafmap
    import tempfile, os
    da = result.grid
    with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as f:
        tmp = f.name
    da.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326").rio.to_raster(tmp)
    m = leafmap.Map(center=[float(da.lat.mean()), float(da.lon.mean())], zoom=6)
    m.add_raster(tmp, colormap="RdYlBu_r", layer_name=da.name or "interpolated")
    display(m)
except ImportError:
    print("Interactive map requires: pip install 'geointerpo[notebooks]'")
    result.plot()
except Exception as e:
    print(f"Map unavailable ({e}); falling back to static plot")
    result.plot()


---
## Part C — 🛰 GEE validation against CHIRPS

Requires: `earthengine authenticate` run once in terminal.

In [ ]:
try:
    import ee  # noqa: F401
    result_gee = Pipeline(
        data='openmeteo',
        variable='precipitation',
        date=DATE,
        boundary=BOUNDARY,
        method=['idw', 'kriging', 'rbf'],
        method_params={'kriging': {'variogram_model': 'gaussian'}},
        resolution=RESOLUTION,
        validate_with_gee=True,        # compare against CHIRPS
    ).run()
    
    print('GEE validation (vs CHIRPS):')
    for k, v in result_gee.gee_metrics.items():
        if k != 'diff_map':
            print(f'  {k}: {v:.3f}')
except ImportError:
    print("GEE section skipped: install earthengine-api and authenticate first")
except Exception as e:
    print(f"GEE error: {e}")


In [ ]:
from geointerpo import viz

fig = viz.plot_diff(
    result_gee.gee_reference,
    result_gee.grids['kriging'],
    title='Kriging − CHIRPS (mm/day)'
)
plt.show()

---
## Save outputs

In [ ]:
result.save('outputs/precipitation', geotiff=True, plot=True)
print('Saved to outputs/precipitation/')